# ZKBrain 跨平台脑机接口 (BCI) 科研算法分析流水线
> **技术支持**：熵基科技脑机事业部 (ZKTeco Brain-Computer Interface Division)
>
> 本 Notebook 专为高校、神经科学实验室（Neuroscience）与算法开发者设计，涵盖：
> 1. 脑电原始数据 (Raw EEG 512Hz) 滤波流水线（0.5-50Hz 带通 + 50Hz 陷波）
> 2. MNE-Python 国际标准脑电结构组装与 10-20 电极（FP1）配置
> 3. Welch PSD (Power Spectral Density) 功率谱密度与 8 频段绝对功率分析
> 4. 专注度/放松度 1Hz 动态特征建模
> 5. 导出符合国际科研标准的 EDF+ / CSV 数据集

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import mne

print(f"MNE 版本: {mne.__version__}")

## 1. 模拟 / 读取 ZKBrain 512Hz 原始脑电信号

In [ ]:
# 采样率 512Hz，采集 10 秒
sfreq = 512
duration = 10
n_samples = sfreq * duration
t = np.linspace(0, duration, n_samples, endpoint=False)

# 构造仿真脑电：包含 10Hz Alpha波、4Hz Theta波与 50Hz 工频干扰
alpha_wave = 25 * np.sin(2 * np.pi * 10 * t)
theta_wave = 15 * np.sin(2 * np.pi * 4 * t)
noise_50hz = 12 * np.sin(2 * np.pi * 50 * t)
pink_noise = np.random.normal(0, 8, n_samples)

raw_eeg = alpha_wave + theta_wave + noise_50hz + pink_noise

plt.figure(figsize=(12, 3))
plt.plot(t[:1024], raw_eeg[:1024], color='#0284c7', lw=1)
plt.title("ZKBrain 原始脑电波形 (Raw EEG - 前 2 秒)")
plt.xlabel("时间 (秒)")
plt.ylabel("微伏 (uV)")
plt.grid(True, alpha=0.3)
plt.show()

## 2. 0.5~50Hz 带通滤波与 50Hz 工频陷波 (Butterworth & IIR Notch)

In [ ]:
# 1. 50Hz 陷波滤波器 (Notch Filter)
b_notch, a_notch = signal.iirnotch(50.0, 30.0, sfreq)
notched_eeg = signal.filtfilt(b_notch, a_notch, raw_eeg)

# 2. 0.5 ~ 50Hz 4阶双向巴特沃斯带通滤波
sos = signal.butter(4, [0.5, 50.0], btype='bandpass', fs=sfreq, output='sos')
filtered_eeg = signal.sosfiltfilt(sos, notched_eeg)

plt.figure(figsize=(12, 4))
plt.plot(t[:1024], raw_eeg[:1024], label='原始信号 (含50Hz工频)', color='#ef4444', alpha=0.6)
plt.plot(t[:1024], filtered_eeg[:1024], label='滤波后信号 (0.5-50Hz Clean EEG)', color='#16a34a', lw=1.5)
plt.title("滤波前后脑电对比")
plt.xlabel("时间 (秒)")
plt.ylabel("幅度 (uV)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. MNE-Python 标准数据封装与 PSD 功率谱密度分析

In [ ]:
# 封装为 MNE RawArray (通道为国际标准 FP1 额贴单导)
ch_names = ['FP1']
ch_types = ['eeg']
info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)

# 将 uV 转换为 V (MNE 标准输入)
data_mne = filtered_eeg.reshape(1, -1) * 1e-6
raw = mne.io.RawArray(data_mne, info)

# 计算 Welch PSD (功率谱密度)
psd = raw.compute_psd(method='welch', fmin=0.5, fmax=50, n_fft=512)
psds, freqs = psd.get_data(return_freqs=True)

plt.figure(figsize=(10, 4))
plt.plot(freqs, 10 * np.log10(psds[0]), color='#7c3aed', lw=2)
plt.axvspan(0.5, 4, color='#e0ffff', alpha=0.4, label='Delta (0.5-4Hz)')
plt.axvspan(4, 8, color='#c7d2fe', alpha=0.4, label='Theta (4-8Hz)')
plt.axvspan(8, 13, color='#fef08a', alpha=0.4, label='Alpha (8-13Hz)')
plt.axvspan(13, 30, color='#fed7aa', alpha=0.4, label='Beta (13-30Hz)')
plt.axvspan(30, 50, color='#fecaca', alpha=0.4, label='Gamma (30-50Hz)')
plt.title("Welch 功率谱密度 (PSD Spectrum) & 8 频段分布")
plt.xlabel("频率 (Hz)")
plt.ylabel("功率谱密度 (dB)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. 导出为国际标准科研数据集 (EDF+ / CSV)

In [ ]:
# 导出为 CSV 时序表
df_export = pd.DataFrame({
    'timestamp_ms': (t * 1000).astype(int),
    'raw_eeg_uV': filtered_eeg,
    'attention': 78,
    'meditation': 65
})
df_export.to_csv('zkbrain_research_dataset.csv', index=False)
print("✅ 已成功生成 zkbrain_research_dataset.csv 科研数据集")

# 导出为欧洲标准脑电格式 EDF+
mne.export.export_raw('zkbrain_research_dataset.edf', raw, fmt='edf', overwrite=True)
print("✅ 已成功生成 zkbrain_research_dataset.edf 标准欧洲脑电文件 (可导入 EEGLAB / BrainVision / Polyman 分析)")